# Random Forest Training - Occupancy Prediction
Trains a random forest classifier to predict bus occupancy level (low, medium, high, very_high).
uses class weights to handle imbalance without discarding data.


In [1]:
# --- setup: mount google drive and sync repo ---
from google.colab import drive
drive.mount('/content/drive')

import os

REPO_PATH = "/content/drive/MyDrive/Occupancy_capstone/occupancy-prediction-capstone"
%cd {REPO_PATH}

!git config --global user.email "eleonorvilla2003@gmail.com"
!git config --global user.name "victoriaeleonor"

!git pull

Mounted at /content/drive
/content/drive/MyDrive/Occupancy_capstone/occupancy-prediction-capstone
Already up to date.


In [2]:
# --- imports ---
import pandas as pd
import numpy as np
import pickle
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

print("Libraries imported successfully")

Libraries imported successfully


In [3]:
# --- configuration ---

# paths to X and y files generated by the feature engineering notebook
DATASET_PATH = "/content/drive/MyDrive/Occupancy_capstone/Dataset"
USE_LAGS = True
suffix   = "with_lags" if USE_LAGS else "no_lags"

X_PATH = f"{DATASET_PATH}/sunt_2024_03_march_{suffix}_X.parquet"
Y_PATH = f"{DATASET_PATH}/sunt_2024_03_march_{suffix}_y.pkl"

# where to save the trained model and results
MODEL_PATH = "/content/drive/MyDrive/Occupancy_capstone/Model"

# train/test split
TEST_SIZE    = 0.2
RANDOM_STATE = 42

# random forest hyperparameters
N_ESTIMATORS     = 100
MAX_DEPTH        = 15
MIN_SAMPLES_SPLIT = 20
MIN_SAMPLES_LEAF  = 10

print(f"X path : {X_PATH}")
print(f"y path : {Y_PATH}")
print(f"model  : {MODEL_PATH}")

X path : /content/drive/MyDrive/Occupancy_capstone/Dataset/sunt_2024_03_march_X.parquet
y path : /content/drive/MyDrive/Occupancy_capstone/Dataset/sunt_2024_03_march_y.pkl
model  : /content/drive/MyDrive/Occupancy_capstone/Model


In [4]:
# --- load data ---

X = pd.read_parquet(X_PATH)
with open(Y_PATH, 'rb') as f:
    y = pickle.load(f)

print("Dataset loaded:")
print(f"  X shape  : {X.shape}")
print(f"  y shape  : {y.shape}")
print(f"  features : {list(X.columns)}")
print(f"\nTarget distribution:")
dist = y.value_counts(normalize=True).sort_index()
for cls, pct in dist.items():
    count = (y == cls).sum()
    bar = '█' * int(pct * 50)
    print(f"  {cls:12s}: {count:>10,} ({pct*100:5.2f}%) {bar}")

Dataset loaded:
  X shape  : (3022798, 11)
  y shape  : (3022798,)
  features : ['route_short_name', 'direction_id', 'pt_sequence', 'stop_id', 'hour', 'day_of_week', 'is_weekend', 'is_rush_hour', 'route_progression', 'loading_lag_1', 'loading_lag_2']

Target distribution:
  high        :    552,506 (18.28%) █████████
  low         :  1,341,277 (44.37%) ██████████████████████
  medium      :    892,718 (29.53%) ██████████████
  very_high   :    236,297 ( 7.82%) ███


In [5]:
# --- analyze class imbalance and compute class weights ---
# random forest accepts class_weight directly in its constructor,
# so no need to compute sample_weight separately like in xgboost

class_counts = y.value_counts()
imbalance_ratio = class_counts.max() / class_counts.min()

print(f"Imbalance ratio : {imbalance_ratio:.1f}:1")
print(f"Majority class  : {class_counts.idxmax()} ({class_counts.max():,})")
print(f"Minority class  : {class_counts.idxmin()} ({class_counts.min():,})")

classes = np.unique(y)
class_weights_array = compute_class_weight('balanced', classes=classes, y=y)
class_weights = dict(zip(classes, class_weights_array))

print(f"\nClass weights:")
for cls, weight in sorted(class_weights.items()):
    print(f"  {cls:12s}: {weight:.4f}")

Imbalance ratio : 5.7:1
Majority class  : low (1,341,277)
Minority class  : very_high (236,297)

Class weights:
  high        : 1.3678
  low         : 0.5634
  medium      : 0.8465
  very_high   : 3.1981


In [6]:
# --- stratified train/test split ---
# stratify=y ensures class distribution is preserved in both sets

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"Train set : {len(X_train):,} records")
print(f"Test set  : {len(X_test):,} records")

print(f"\nTrain distribution:")
for cls, pct in y_train.value_counts(normalize=True).sort_index().items():
    print(f"  {cls:12s}: {pct*100:.2f}%")

print(f"\nTest distribution:")
for cls, pct in y_test.value_counts(normalize=True).sort_index().items():
    print(f"  {cls:12s}: {pct*100:.2f}%")

Train set : 2,418,238 records
Test set  : 604,560 records

Train distribution:
  high        : 18.28%
  low         : 44.37%
  medium      : 29.53%
  very_high   : 7.82%

Test distribution:
  high        : 18.28%
  low         : 44.37%
  medium      : 29.53%
  very_high   : 7.82%


In [7]:
# --- train random forest model ---

model = RandomForestClassifier(
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    min_samples_split=MIN_SAMPLES_SPLIT,
    min_samples_leaf=MIN_SAMPLES_LEAF,
    # class_weight passed directly: rf handles weighting internally
    class_weight=class_weights,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

print("Random Forest configuration:")
print(f"  n_estimators      : {N_ESTIMATORS}")
print(f"  max_depth         : {MAX_DEPTH}")
print(f"  min_samples_split : {MIN_SAMPLES_SPLIT}")
print(f"  min_samples_leaf  : {MIN_SAMPLES_LEAF}")
print(f"  class_weight      : balanced (custom)")
print("\nTraining...")

start_time = time.time()
model.fit(X_train, y_train)
elapsed = time.time() - start_time

print(f"\nTraining completed in {elapsed:.2f}s ({elapsed/60:.1f} min)")

Random Forest configuration:
  n_estimators      : 100
  max_depth         : 15
  min_samples_split : 20
  min_samples_leaf  : 10
  class_weight      : balanced (custom)

Training...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done  46 tasks      | elapsed:  5.9min



Training completed in 711.25s (11.9 min)


[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed: 11.8min finished


In [8]:
# --- evaluate model ---
# primary metric: f1 macro (treats all classes equally regardless of size)
# secondary metric: balanced accuracy (average recall per class)

y_pred_train = model.predict(X_train)
y_pred_test  = model.predict(X_test)

metrics = {
    'accuracy'         : (accuracy_score(y_train, y_pred_train),          accuracy_score(y_test, y_pred_test)),
    'balanced accuracy': (balanced_accuracy_score(y_train, y_pred_train),  balanced_accuracy_score(y_test, y_pred_test)),
    'f1 macro'         : (f1_score(y_train, y_pred_train, average='macro'), f1_score(y_test, y_pred_test, average='macro')),
}

print(f"{'Metric':<22} {'Train':>10} {'Test':>10} {'Gap':>10}")
print('-' * 55)
for name, (train_val, test_val) in metrics.items():
    print(f"{name:<22} {train_val:>10.4f} {test_val:>10.4f} {train_val - test_val:>10.4f}")

gap = metrics['balanced accuracy'][0] - metrics['balanced accuracy'][1]
if gap > 0.1:
    print(f"\nWarning: high gap ({gap:.2%}) — possible overfitting")
elif gap > 0.05:
    print(f"\nModerate gap ({gap:.2%}) — acceptable")
else:
    print(f"\nLow gap ({gap:.2%}) — model generalizes well")

[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:   17.4s
[Parallel(n_jobs=2)]: Done 100 out of 100 | elapsed:   37.6s finished
[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    5.4s
[Parallel(n_jobs=2)]: Done 100 out of 100 | elapsed:   10.4s finished


Metric                      Train       Test        Gap
-------------------------------------------------------
accuracy                   0.9356     0.9331     0.0026
balanced accuracy          0.9307     0.9273     0.0034
f1 macro                   0.9284     0.9252     0.0032

Low gap (0.34%) — model generalizes well


In [9]:
# --- classification report ---
print("Classification Report (test set):")
print(classification_report(y_test, y_pred_test))

Classification Report (test set):
              precision    recall  f1-score   support

        high       0.91      0.90      0.90    110501
         low       0.96      0.96      0.96    268256
      medium       0.91      0.90      0.91    178544
   very_high       0.91      0.94      0.93     47259

    accuracy                           0.93    604560
   macro avg       0.92      0.93      0.93    604560
weighted avg       0.93      0.93      0.93    604560



In [10]:
# --- confusion matrix ---

labels = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred_test, labels=labels)

print("Confusion Matrix (test set):")
print(f"\n{'':>12}", end='')
for label in labels:
    print(f"{label:>12}", end='')
print("  <- predicted")
print('-' * (12 + 12 * len(labels)))
for i, label in enumerate(labels):
    print(f"{label:>12}", end='')
    for j in range(len(labels)):
        print(f"{cm[i,j]:>12,}", end='')
    print("  | actual")

print("\nPer-class accuracy:")
for i, label in enumerate(labels):
    total   = cm[i, :].sum()
    correct = cm[i, i]
    acc     = correct / total if total > 0 else 0
    status  = 'OK' if acc > 0.6 else 'LOW' if acc > 0.4 else 'POOR'
    print(f"  {status:4s} {label:12s}: {acc:.2%} ({correct:,}/{total:,})")

Confusion Matrix (test set):

                    high         low      medium   very_high  <- predicted
------------------------------------------------------------
        high      99,747          95       6,543       4,116  | actual
         low         117     258,623       9,506          10  | actual
      medium       7,333       9,804     161,366          41  | actual
   very_high       2,848           5          42      44,364  | actual

Per-class accuracy:
  OK   high        : 90.27% (99,747/110,501)
  OK   low         : 96.41% (258,623/268,256)
  OK   medium      : 90.38% (161,366/178,544)
  OK   very_high   : 93.87% (44,364/47,259)


In [11]:
# --- feature importance ---

importances   = model.feature_importances_
feature_names = X.columns.tolist()
indices       = np.argsort(importances)[::-1]

print("Top features by importance:")
print('-' * 55)
for i in range(min(15, len(feature_names))):
    idx = indices[i]
    bar = '█' * int(importances[idx] * 50)
    print(f"{i+1:2d}. {feature_names[idx]:<25} {importances[idx]:.4f}  {bar}")

cumsum = 0
for i, idx in enumerate(indices):
    cumsum += importances[idx]
    if cumsum >= 0.8:
        print(f"\nTop {i+1} features explain 80% of total importance")
        break

Top features by importance:
-------------------------------------------------------
 1. loading_lag_1             0.5730  ████████████████████████████
 2. loading_lag_2             0.3557  █████████████████
 3. hour                      0.0260  █
 4. route_progression         0.0184  
 5. pt_sequence               0.0107  
 6. route_short_name          0.0062  
 7. stop_id                   0.0050  
 8. day_of_week               0.0015  
 9. direction_id              0.0013  
10. is_weekend                0.0013  
11. is_rush_hour              0.0010  

Top 2 features explain 80% of total importance


In [12]:
# --- compare with baseline and xgboost ---
# baseline: always predict the majority class (simplest possible model)
# xgboost results are loaded from file if available (saved by the xgboost training notebook)
# otherwise falls back to hardcoded values from the original run

majority_class = y_train.value_counts().idxmax()
baseline_acc   = (y_test == majority_class).mean()

XGB_RESULTS_PATH = f"{MODEL_PATH}/xgb_results_{suffix}.pkl"
if os.path.exists(XGB_RESULTS_PATH):
    with open(XGB_RESULTS_PATH, 'rb') as f:
        xgb_results = pickle.load(f)
    xgb_acc      = xgb_results['accuracy']
    xgb_bal_acc  = xgb_results['balanced_accuracy']
    xgb_f1_macro = xgb_results['f1_macro']
    print("XGBoost results loaded from file")
else:
    xgb_acc      = 0.6172
    xgb_bal_acc  = 0.5894
    xgb_f1_macro = 0.5031
    print("XGBoost results using hardcoded values (run TrainingXGBoost first to update)")

rf_acc      = metrics['accuracy'][1]
rf_bal_acc  = metrics['balanced accuracy'][1]
rf_f1_macro = metrics['f1 macro'][1]

print(f"\n{'Model':<20} {'Accuracy':>10} {'Bal. Acc':>10} {'F1 Macro':>10}")
print('-' * 55)
print(f"{'Baseline':<20} {baseline_acc:>10.4f} {'—':>10} {'—':>10}")
print(f"{'Random Forest':<20} {rf_acc:>10.4f} {rf_bal_acc:>10.4f} {rf_f1_macro:>10.4f}")
print(f"{'XGBoost':<20} {xgb_acc:>10.4f} {xgb_bal_acc:>10.4f} {xgb_f1_macro:>10.4f}")
print(f"\nRandom Forest vs XGBoost:")
print(f"  balanced accuracy : {(rf_bal_acc - xgb_bal_acc)*100:+.2f}%")
print(f"  f1 macro          : {(rf_f1_macro - xgb_f1_macro)*100:+.2f}%")

XGBoost results using hardcoded values (run TrainingXGBoost first to update)

Model                  Accuracy   Bal. Acc   F1 Macro
-------------------------------------------------------
Baseline                 0.4437          —          —
Random Forest            0.9331     0.9273     0.9252
XGBoost                  0.6172     0.5894     0.5031

Random Forest vs XGBoost:
  balanced accuracy : +33.79%
  f1 macro          : +42.21%


In [13]:
# --- save model and results to drive ---

model_file = f"{MODEL_PATH}/rf_occupancy_{suffix}.pkl"
with open(model_file, 'wb') as f:
    pickle.dump(model, f)
print(f"Model saved      -> {model_file}")

features_file = f"{MODEL_PATH}/rf_feature_names_{suffix}.pkl"
with open(features_file, 'wb') as f:
    pickle.dump(feature_names, f)
print(f"Feature names    -> {features_file}")

rf_results = {
    'accuracy'         : rf_acc,
    'balanced_accuracy': rf_bal_acc,
    'f1_macro'         : rf_f1_macro,
}
rf_results_file = f"{MODEL_PATH}/rf_results_{suffix}.pkl"
with open(rf_results_file, 'wb') as f:
    pickle.dump(rf_results, f)
print(f"RF results       -> {rf_results_file}")

Model saved      -> /content/drive/MyDrive/Occupancy_capstone/Model/rf_occupancy.pkl
Feature names    -> /content/drive/MyDrive/Occupancy_capstone/Model/rf_feature_names.pkl
RF results       -> /content/drive/MyDrive/Occupancy_capstone/Model/rf_results.pkl


In [14]:
# --- push changes to github ---
!git add .
!git commit -m "update: rf training with drive saving, consistent metrics and xgboost comparison"
!git push

[main 07fdc94] update: rf training with drive saving, consistent metrics and xgboost comparison
 1 file changed, 1 insertion(+)
 create mode 100644 Occupancy_dataloader_SUNT_OD (1).ipynb
Enumerating objects: 4, done.
Counting objects: 100% (4/4), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 4.94 KiB | 389.00 KiB/s, done.
Total 3 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/victoriaeleonor/occupancy-prediction-capstone.git
   c79f24e..07fdc94  main -> main
